In [2]:
import ccxt
import talib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from backtesting.lib import crossover,cross
from backtesting import Backtest, Strategy

pd.set_option('display.max_columns', None)
%matplotlib inline

In [15]:
def fetch_ohlcv(symbol='BTC/USDT', timeframe='15m', limit=1500, exchange_id='binance'):
    exchange_class = getattr(ccxt, exchange_id)
    exchange = exchange_class({'enableRateLimit': True})
    raw = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)
    
    # backtesting.py تتطلب أسماء أعمدة معينة وحروف كبيرة (Capitalized)
    df = pd.DataFrame(raw, columns=['Date', 'Open', 'High', 'Low', 'Close', 'Volume'])
    df['Date'] = pd.to_datetime(df['Date'], unit='ms')
    df.set_index('Date', inplace=True)
    return df

# إطار زمني 15 دقيقة (ضمن النطاق 5م - 30م+)
SYMBOL = 'LA/USDT'
TIMEFRAME = '30m'
data = fetch_ohlcv(symbol=SYMBOL, timeframe=TIMEFRAME, limit=500, exchange_id='bybit')
print(f'تم جلب {len(data)} شمعة لـ {SYMBOL} على إطار {TIMEFRAME}')
data.tail()


تم جلب 500 شمعة لـ LA/USDT على إطار 30m


,Open,High,Low,Close,Volume
Date,,,,,
2026-07-29 17:00:00,0.05349,0.05497,0.05322,0.05492,413229.0
2026-07-29 17:30:00,0.05492,0.05558,0.05490,0.05514,100637.8
2026-07-29 18:00:00,0.05514,0.05544,0.05383,0.05408,397249.8
2026-07-29 18:30:00,0.05408,0.05408,0.05324,0.05359,126593.7
2026-07-29 19:00:00,0.05359,0.05412,0.05359,0.05399,39991.0


In [16]:
import talib
import numpy as np
import pandas as pd
from backtesting import Strategy, Backtest

# ==========================================
# 1. Hull MA Helper Functions (Matching Pine Script)
# ==========================================

def compute_hma(series, length):
    half_length = int(length / 2)
    sqrt_length = int(np.round(np.sqrt(length)))
    wma_half = talib.WMA(series, half_length)
    wma_full = talib.WMA(series, length)
    return talib.WMA(2 * wma_half - wma_full, sqrt_length)

def compute_ehma(series, length):
    half_length = int(length / 2)
    sqrt_length = int(np.round(np.sqrt(length)))
    ema_half = talib.EMA(series, half_length)
    ema_full = talib.EMA(series, length)
    return talib.EMA(2 * ema_half - ema_full, sqrt_length)

def compute_thma(series, length):
    third_length = int(length / 3)
    half_length = int(length / 2)
    wma_third = talib.WMA(series, third_length)
    wma_half = talib.WMA(series, half_length)
    wma_full = talib.WMA(series, length)
    return talib.WMA(3 * wma_third - wma_half - wma_full, length)

# ==========================================
# 2. Unified Indicator Calculation Function
# ==========================================

def compute_all_indicators(df, bb_period=20, bb_dev=2.0, atr_period=14, atr_mult=1.0, 
                           mom_period=10, hull_mode='Hma', hull_length=55, hull_mult=1.0):
    """
    دالة موحدة لحساب مؤشرات FLI, MOM, و Hull MA وإضافتها للبيانات
    """
    close = df['Close'].to_numpy(dtype=float)
    high = df['High'].to_numpy(dtype=float)
    low = df['Low'].to_numpy(dtype=float)

    # 1. حساب Bollinger Bands و ATR لـ FLI
    upper, mid, lower = talib.BBANDS(close, timeperiod=bb_period, nbdevup=bb_dev, nbdevdn=bb_dev)
    atr = talib.ATR(high, low, close, timeperiod=atr_period)

    # 2. حساب مؤشر MOM
    df['MOM'] = talib.MOM(close, timeperiod=mom_period)

    # 3. حساب خط المتابعة FollowLine (FLI)
    fl = np.full_like(close, np.nan)
    trend = np.zeros(len(close), dtype=int)  # 1 للأخضر/الأزرق (صعود), -1 للأحمر (هبوط)

    curr_trend = 0
    curr_fl = np.nan

    for i in range(len(close)):
        if np.isnan(upper[i]) or np.isnan(atr[i]):
            continue

        if curr_trend == 0:
            if close[i] > upper[i]: 
                curr_trend = 1
            elif close[i] < lower[i]: 
                curr_trend = -1

        if curr_trend == 1:
            level = low[i] - (atr[i] * atr_mult)
            curr_fl = max(curr_fl, level) if not np.isnan(curr_fl) else level
            if close[i] < curr_fl:
                curr_trend = 0
                curr_fl = np.nan

        elif curr_trend == -1:
            level = high[i] + (atr[i] * atr_mult)
            curr_fl = min(curr_fl, level) if not np.isnan(curr_fl) else level
            if close[i] > curr_fl:
                curr_trend = 0
                curr_fl = np.nan

        trend[i] = curr_trend
        fl[i] = curr_fl

    df['FollowLine'] = fl
    df['FollowTrend'] = trend

    # 4. حساب مؤشر Hull MA بناءً على الوضع المختار
    adj_length = int(hull_length * hull_mult)
    if hull_mode == 'Hma':
        df['HullMA'] = compute_hma(close, adj_length)
    elif hull_mode == 'Ehma':
        df['HullMA'] = compute_ehma(close, adj_length)
    elif hull_mode == 'Thma':
        df['HullMA'] = compute_thma(close, adj_length)
    else:
        df['HullMA'] = np.nan

    # إزاحة المؤشر بمقدار 2 شمعة لمطابقة منطق Pine Script (SHULL)
    df['HullMA_Shifted'] = df['HullMA'].shift(2)

    return df

# ==========================================
# 3. Updated Combined Strategy with Hull MA
# ==========================================

class CombinedMomFliHullStrategy(Strategy):
    def init(self):
        # تسجيل المؤشرات ليتمكن backtesting.py من قراءتها كسلاسل زمنية
        self.mom = self.I(lambda x: x, self.data.MOM, name='MOM')
        self.fli_trend = self.I(lambda x: x, self.data.FollowTrend, name='FollowTrend')
        self.fli_line = self.I(lambda x: x, self.data.FollowLine, name='FollowLine')
        self.hull_ma = self.I(lambda x: x, self.data.HullMA, name='HullMA')
        self.hull_ma_shifted = self.I(lambda x: x, self.data.HullMA_Shifted, name='HullMA_Shifted')

        # إعدادات MomStrategy
        self.up_zone = 0.00639
        self.middle_zone = 0.00001
        self.down_zone = -0.006660

    def next(self):
        # التأكد من توفر بيانات كافية للحساب
        if len(self.fli_trend) < 3:
            return

        mom_now = self.mom[-1]
        fli_trend_now = self.fli_trend[-1]
        fli_trend_prev = self.fli_trend[-2]
        
        hull_now = self.hull_ma[-1]
        hull_shifted = self.hull_ma_shifted[-1]
        hull_prev = self.hull_ma[-2]
        hull_shifted_prev = self.hull_ma_shifted[-2]
        
        # تحديد اتجاه Hull MA (1 للصعود، -1 للهبوط) بناءً على MHULL > SHULL
        hull_trend = 1 if hull_now > hull_shifted else -1
        hull_trend_prev = 1 if hull_prev > hull_shifted_prev else -1

        # --- استخراج إشارات MOM ---
        mom_buy = mom_now > self.up_zone
        mom_sell = mom_now < self.down_zone
        mom_exit_long = mom_now < self.middle_zone
        mom_exit_short = mom_now > self.middle_zone

        # --- استخراج إشارات FLI ---
        fli_buy = (fli_trend_now == 1 and fli_trend_prev != 1)
        fli_sell = (fli_trend_now == -1 and fli_trend_prev != -1)
        fli_exit_long = (fli_trend_now != 1 and fli_trend_prev == 1)
        fli_exit_short = (fli_trend_now != -1 and fli_trend_prev == -1)

        # --- استخراج إشارات Hull MA (تقاطع) ---
        hull_buy = (hull_trend == 1 and hull_trend_prev != 1)
        hull_sell = (hull_trend == -1 and hull_trend_prev != -1)
        hull_exit_long = (hull_trend == -1 and hull_trend_prev == 1)
        hull_exit_short = (hull_trend == 1 and hull_trend_prev == -1)

        # --- 1. منطق الدخول (شراء Long) ---
        # الدخول إذا أعطى MOM أو FLI أو Hull إشارة شراء، مع التأكد من أن اتجاه Hull العام صعودي (فلتر)
        if (mom_buy or (fli_buy and not mom_sell) or hull_buy) and hull_trend == 1:
            if self.position.is_short:
                self.position.close()  # إغلاق الشورت فوراً
            if not self.position.is_long:
                self.buy()  # فتح لونغ

        # --- 2. منطق الدخول (بيع Short) ---
        # الدخول إذا أعطى MOM أو FLI أو Hull إشارة بيع، مع التأكد من أن اتجاه Hull العام هبوطي (فلتر)
        elif (mom_sell or (fli_sell and not mom_buy) or hull_sell) and hull_trend == -1:
            if self.position.is_long:
                self.position.close()  # إغلاق اللونغ فوراً
            if not self.position.is_short:
                self.sell()  # فتح شورت

        # --- 3. منطق الخروج (إغلاق الصفقة) ---
        # إذا لم تكن هناك إشارات دخول جديدة، نتحقق من إشارات الخروج من أي استراتيجية
        else:
            if self.position.is_long:
                if mom_exit_long or fli_exit_long or hull_exit_long:
                    self.position.close()
            elif self.position.is_short:
                if mom_exit_short or fli_exit_short or hull_exit_short:
                    self.position.close()

# ==========================================
# 4. Example Usage (How to run it)
# ==========================================
if __name__ == "__main__":
    # 1. افترض أن 'data' هو DataFrame الخاص بك الذي تم جلبه مسبقاً
    # data = fetch_ohlcv(symbol='BTC/USDT', timeframe='15m', limit=1500)
    
    # 2. حساب جميع المؤشرات (يمكنك تغيير hull_mode إلى 'Ehma' أو 'Thma' حسب رغبتك)
    data = compute_all_indicators(data, hull_mode='Hma', hull_length=55, hull_mult=1.0)
    
    # 3. تشغيل الاختبار الخلفي
    # ملاحظة: exclusive_orders=True تضمن إغلاق أي صفقة مفتوحة فور فتح صفقة عكسية
    bt = Backtest(data, CombinedMomFliHullStrategy, cash=100, commission=0.0001, exclusive_orders=True)
    
    # 4. عرض النتائج والرسم البياني
    stats = bt.run()
    print(stats)
    bt.plot()

Backtest.run:   0%|          | 0/437 [00:00<?, ?bar/s]

Start                     2026-07-19 09:30:00
End                       2026-07-29 19:00:00
Duration                     10 days 09:30:00
Exposure Time [%]                        47.6
Equity Final [$]                     88.18507
Equity Peak [$]                     136.88823
Commissions [$]                       0.50747
Return [%]                          -11.81493
Buy & Hold Return [%]                 8.34838
Return (Ann.) [%]                   -98.45792
Volatility (Ann.) [%]                 6.63206
CAGR [%]                            -98.78994
Sharpe Ratio                        -14.84576
Sortino Ratio                        -0.86686
Calmar Ratio                          -2.3099
Alpha [%]                           -16.89824
Beta                                   0.6089
Max. Drawdown [%]                    -42.6244
Avg. Drawdown [%]                   -36.33083
Max. Drawdown Duration        6 days 13:00:00
Avg. Drawdown Duration        4 days 06:45:00
# Trades                          